In [2]:
!pip install -q groq python-dotenv pydantic

In [3]:
import os
import json

from groq import Groq
from pydantic import BaseModel, Field, ValidationError
from typing import List

In [4]:
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API key: ")

client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully.")

Enter your Groq API key: ········
Groq client initialized successfully.


In [5]:
MODEL_NAME = "openai/gpt-oss-20b"

In [6]:
class CriterionResult(BaseModel):
    criterion: str
    achieved: bool
    marks_awarded: float = Field(ge=0)
    explanation: str


class EvaluationResult(BaseModel):
    criteria: List[CriterionResult]

    total_marks: float = Field(ge=0)
    maximum_marks: float = Field(gt=0)

    overall_feedback: str

    strengths: List[str]
    areas_for_improvement: List[str]

In [12]:
EVALUATION_PROMPT = """
You are an AI-based academic answer evaluator designed specifically
for CBSE students studying in Classes 9 and 10.

Your task is to evaluate the student's answer fairly and consistently
using the question, reference answer, marking scheme, and maximum marks.

EVALUATION RULES:

1. Evaluate conceptual correctness rather than exact wording.

2. Do not penalize the student simply because their wording differs
   from the reference answer.

3. Accept alternative explanations if they are academically correct.

4. Award partial marks when the student demonstrates partial
   understanding.

5. Identify incorrect concepts or factual misconceptions.

6. Do not penalize minor spelling or grammatical errors if the
   intended meaning is clear.

7. Evaluate according to the expected knowledge level of a
   CBSE Class 9-10 student.

8. Do not expect knowledge beyond the provided curriculum context.

9. Award marks only for relevant and correct information.

10. The total marks awarded must NEVER exceed the maximum marks.

11. Follow the provided marking scheme whenever one is available.

12. Return ONLY valid JSON. Do not include markdown formatting,
    ```json blocks, or additional text outside the JSON.


SUBJECT:
{subject}

TOPIC:
{topic}

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

MARKING SCHEME:
{marking_scheme}

MAXIMUM MARKS:
{maximum_marks}

STUDENT ANSWER:
{student_answer}


Return the evaluation using exactly this JSON structure:

{{
    "criteria": [
        {{
            "criterion": "Criterion being evaluated",
            "achieved": true,
            "marks_awarded": 1,
            "explanation": "Reason for awarding these marks"
        }}
    ],

    "total_marks": 0,
    "maximum_marks": {maximum_marks},

    "overall_feedback": "Overall evaluation of the answer",

    "strengths": [
        "A strength of the answer"
    ],

    "areas_for_improvement": [
        "An area that could be improved"
    ]
}}
"""

In [14]:
def build_evaluation_prompt(
    question,
    student_answer,
    reference_answer,
    marking_scheme,
    maximum_marks,
    subject="Not specified",
    topic="Not specified"
):

    prompt = EVALUATION_PROMPT.format(
        subject=subject,
        topic=topic,
        question=question,
        reference_answer=reference_answer,
        marking_scheme=marking_scheme,
        maximum_marks=maximum_marks,
        student_answer=student_answer
    )

    return prompt

In [8]:
def call_llm(prompt):
    """
    Sends a prompt to the Groq LLM and returns
    the generated response as text.
    """

    if not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    try:

        response = client.chat.completions.create(
            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a fair and consistent academic "
                        "answer-sheet evaluator."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.1
        )

        return response.choices[0].message.content

    except Exception as e:

        print("LLM API Error:")
        print(e)

        return None

In [9]:
def parse_evaluation(response_text):

    if response_text is None:
        raise ValueError("No response received from LLM.")

    try:

        # Convert JSON string -> Python dictionary
        data = json.loads(response_text)

        # Validate structure
        evaluation = EvaluationResult(**data)

        # Validate maximum marks
        if evaluation.total_marks > evaluation.maximum_marks:
            raise ValueError(
                "LLM awarded marks greater than maximum marks."
            )

        # Validate criterion total
        criterion_total = sum(
            criterion.marks_awarded
            for criterion in evaluation.criteria
        )

        if abs(criterion_total - evaluation.total_marks) > 0.01:
            raise ValueError(
                "Criterion marks do not match total marks."
            )

        return evaluation

    except json.JSONDecodeError as e:

        print("Invalid JSON returned by LLM:")
        print(response_text)

        raise e

    except ValidationError as e:

        print("LLM response does not match required schema:")
        print(e)

        raise e

In [10]:
def evaluate_answer(
    question,
    student_answer,
    reference_answer,
    marking_scheme,
    maximum_marks,
    subject="Not specified",
    topic="Not specified"
):

    if not student_answer.strip():
        raise ValueError("Student answer cannot be empty.")

    if maximum_marks <= 0:
        raise ValueError("Maximum marks must be greater than 0.")

    # Build prompt
    prompt = build_evaluation_prompt(
        question=question,
        student_answer=student_answer,
        reference_answer=reference_answer,
        marking_scheme=marking_scheme,
        maximum_marks=maximum_marks,
        subject=subject,
        topic=topic
    )

    # Send prompt to Groq
    response = call_llm(prompt)

    # Parse + validate response
    evaluation = parse_evaluation(response)

    return evaluation

In [15]:
question = """
What is photosynthesis?
"""

reference_answer = """
Photosynthesis is the process by which green plants use sunlight,
carbon dioxide and water to produce glucose. Oxygen is released
as a by-product.
"""

student_answer = """
Photosynthesis is the process where plants use sunlight,
water and carbon dioxide to make their food and release oxygen.
"""

marking_scheme = """
1 mark - states that plants prepare food/glucose
1 mark - mentions sunlight
1 mark - mentions carbon dioxide and water
1 mark - mentions oxygen is released
"""

result = evaluate_answer(
    subject="Science",
    topic="Photosynthesis",

    question=question,

    student_answer=student_answer,

    reference_answer=reference_answer,

    marking_scheme=marking_scheme,

    maximum_marks=4
)

print(result)

criteria=[CriterionResult(criterion='States that plants prepare food/glucose', achieved=True, marks_awarded=1.0, explanation='The answer says "make their food", which indicates glucose production.'), CriterionResult(criterion='Mentions sunlight', achieved=True, marks_awarded=1.0, explanation='The answer explicitly states "use sunlight".'), CriterionResult(criterion='Mentions carbon dioxide and water', achieved=True, marks_awarded=1.0, explanation='Both "water" and "carbon dioxide" are mentioned.'), CriterionResult(criterion='Mentions oxygen is released', achieved=True, marks_awarded=1.0, explanation='The answer includes "release oxygen".')] total_marks=4.0 maximum_marks=4.0 overall_feedback='Excellent answer, fully meets all criteria.' strengths=['Clear and concise explanation covering all key points.'] areas_for_improvement=[]


In [16]:
print(f"Marks: {result.total_marks}/{result.maximum_marks}")

print("\nCriterion-wise Evaluation:")

for criterion in result.criteria:

    print("\nCriterion:", criterion.criterion)
    print("Achieved:", criterion.achieved)
    print("Marks:", criterion.marks_awarded)
    print("Explanation:", criterion.explanation)


print("\nOverall Feedback:")
print(result.overall_feedback)


print("\nStrengths:")

for strength in result.strengths:
    print("-", strength)


print("\nAreas for Improvement:")

for improvement in result.areas_for_improvement:
    print("-", improvement)

Marks: 4.0/4.0

Criterion-wise Evaluation:

Criterion: States that plants prepare food/glucose
Achieved: True
Marks: 1.0
Explanation: The answer says "make their food", which indicates glucose production.

Criterion: Mentions sunlight
Achieved: True
Marks: 1.0
Explanation: The answer explicitly states "use sunlight".

Criterion: Mentions carbon dioxide and water
Achieved: True
Marks: 1.0
Explanation: Both "water" and "carbon dioxide" are mentioned.

Criterion: Mentions oxygen is released
Achieved: True
Marks: 1.0
Explanation: The answer includes "release oxygen".

Overall Feedback:
Excellent answer, fully meets all criteria.

Strengths:
- Clear and concise explanation covering all key points.

Areas for Improvement:
